In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd

ticker = "SQM-B.SN"   # cambiar a elección
data = yf.download(ticker, period="6mo", interval="1d", auto_adjust=True)
close = data["Close"].dropna()

# retornos logarítmicos diarios
log_ret = np.log(close / close.shift(1)).dropna()

# volatilidad histórica anualizada (asumiendo 252 días hábiles)
sigma_hist = (log_ret.std() * np.sqrt(252)).item()
S0 = close.iloc[-1].item()

print(f"S0 = {S0:.2f}, sigma anualizada = {sigma_hist:.2%}")

# Paso 2 — Parámetros de la opción

In [ ]:
K = round(S0 / 10) * 10     # strike ATM redondeado, ajusta según tu acción
T = 0.25                     # 3 meses
r = 0.06                     # tasa libre de riesgo CLP corta
sigma = sigma_hist
# Usar volitilidad histórica como estimado volatilidad de opción

# Paso 3 — Black-Scholes cerrado

In [ ]:
from scipy.stats import norm

def bs_price(S, K, T, r, sigma, option_type="call"):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    elif option_type == "put":
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
    else:
        raise ValueError("option_type debe ser 'call' o 'put'")

c_bs = bs_price(S0, K, T, r, sigma, "call")
p_bs = bs_price(S0, K, T, r, sigma, "put")


## verificar paridad put-call

lhs = c_bs + K * np.exp(-r * T)
rhs = p_bs + S0
print(lhs, rhs)   

### Paso 4 — Árbol binomial CRR

In [ ]:
def binomial_price(S0, K, T, r, sigma, n, option_type="call", american=False):
    dt = T / n
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)
    disc = np.exp(-r * dt)

    # precios de la acción en el vencimiento
    j = np.arange(n + 1)
    ST = S0 * u**j * d**(n - j)

    if option_type == "call":
        values = np.maximum(ST - K, 0.0)
    else:
        values = np.maximum(K - ST, 0.0)

    # retroceso en el árbol
    for i in range(n - 1, -1, -1):
        values = disc * (p * values[1:i+2] + (1 - p) * values[0:i+1])
        if american:
            Si = S0 * u**np.arange(i + 1) * d**(i - np.arange(i + 1))
            if option_type == "call":
                exercise = np.maximum(Si - K, 0.0)
            else:
                exercise = np.maximum(K - Si, 0.0)
            values = np.maximum(values, exercise)

    return values[0]

c_bin = binomial_price(S0, K, T, r, sigma, n=200, option_type="call")
p_bin = binomial_price(S0, K, T, r, sigma, n=200, option_type="put")
p_bin_american = binomial_price(S0, K, T, r, sigma, n=200, option_type="put", american=True)

# Paso 5 — Convergencia del árbol a Black-Scholes

In [ ]:
import matplotlib.pyplot as plt

ns = [5, 10, 25, 50, 100, 250, 500]
prices = [binomial_price(S0, K, T, r, sigma, n, "call") for n in ns]

plt.figure(figsize=(7,4))
plt.plot(ns, prices, marker="o", label="Binomial")
plt.axhline(c_bs, color="red", linestyle="--", label="Black-Scholes")
plt.xlabel("n (pasos del árbol)")
plt.ylabel("Precio call")
plt.legend()
plt.title(f"Convergencia del árbol binomial — {ticker}")
plt.show()

# Paso 6 — Griegas analíticas (Black-Scholes)

In [ ]:
def bs_greeks(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    delta_call = norm.cdf(d1)
    delta_put = delta_call - 1
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))     # igual para call y put
    vega = S * norm.pdf(d1) * np.sqrt(T)                 # igual para call y put (por 1% mover /100)

    return {
        "delta_call": delta_call,
        "delta_put": delta_put,
        "gamma": gamma,
        "vega": vega,
    }

greeks = bs_greeks(S0, K, T, r, sigma)
print(greeks)

# Paso 7 — Delta y Gamma numéricos desde el árbol (diferencias finitas)

In [ ]:
def numeric_delta_gamma(pricer, S0, K, T, r, sigma, n, h=0.5, option_type="call"):
    up = pricer(S0 + h, K, T, r, sigma, n, option_type)
    mid = pricer(S0, K, T, r, sigma, n, option_type)
    down = pricer(S0 - h, K, T, r, sigma, n, option_type)
    delta = (up - down) / (2 * h)
    gamma = (up - 2 * mid + down) / (h**2)
    return delta, gamma

delta_num, gamma_num = numeric_delta_gamma(binomial_price, S0, K, T, r, sigma, n=200)

# Paso 8 — Curva de Delta vs. Spot

In [ ]:
S_range = np.linspace(0.5 * S0, 1.5 * S0, 60)
deltas = [bs_greeks(S, K, T, r, sigma)["delta_call"] for S in S_range]

plt.plot(S_range, deltas)
plt.axvline(S0, color="gray", linestyle=":", label="S0 actual")
plt.xlabel("Precio del subyacente")
plt.ylabel("Delta de la call")
plt.legend()
plt.show()

# Paso 9 — Cobertura delta-neutral (la parte de "sales/trading")

In [ ]:
n_options_sold = 1000
shares_to_buy = n_options_sold * greeks["delta_call"]
print(f"Para quedar delta-neutral debo comprar ≈ {shares_to_buy:,.0f} acciones")